# 🎯 Hallucination Hunter - Training Notebook

Train Qwen2.5-7B to detect hallucinations using GRPO (Group Relative Policy Optimization)

**Environment**: [HuggingFace Space](https://huggingface.co/spaces/tusharpawar21/hallicunation-Hunt)

**Model**: Qwen/Qwen2.5-7B-Instruct (with Unsloth)

**Training**: TRL GRPO Trainer

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q unsloth trl transformers datasets accelerate bitsandbytes httpx matplotlib pandas

In [ ]:
import torch
import httpx
import json
import matplotlib.pyplot as plt
import pandas as pd
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainingArguments
from datasets import Dataset
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Environment Client

In [ ]:
class HallucinationEnvClient:
    """Client for Hallucination Hunter environment"""
    
    def __init__(self, base_url: str):
        self.base_url = base_url.rstrip('/')
        self.client = httpx.Client(timeout=30.0)
        self.current_state = None
        
    def reset(self):
        """Reset environment and get initial observation"""
        response = self.client.post(f"{self.base_url}/reset")
        response.raise_for_status()
        data = response.json()
        self.current_state = data
        return data['observation'], data['info']
    
    def step(self, action: str):
        """Take action and get reward"""
        response = self.client.post(
            f"{self.base_url}/step",
            json={"action": action}
        )
        response.raise_for_status()
        data = response.json()
        return data['observation'], data['reward'], data['done'], data['info']
    
    def health(self):
        """Check environment health"""
        response = self.client.get(f"{self.base_url}/health")
        response.raise_for_status()
        return response.json()

# Initialize environment
ENV_URL = "https://tusharpawar21-hallicunation-hunt.hf.space"
env = HallucinationEnvClient(ENV_URL)

# Test connection
health = env.health()
print(f"✅ Connected to environment")
print(f"Episodes available: {health['episode_count']}")
print(f"Difficulty distribution: {health['difficulty_distribution']}")

## 3. Load Model with Unsloth

In [ ]:
# Load Qwen2.5-7B with Unsloth (4-bit quantization)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    dtype=None,  # Auto-detect
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✅ Model loaded with LoRA adapters")

## 4. Reward Function

In [ ]:
def compute_reward(generated_text: str, observation: dict) -> float:
    """Compute reward by sending action to environment"""
    try:
        _, reward, done, info = env.step(generated_text)
        return reward
    except Exception as e:
        print(f"Error computing reward: {e}")
        return -1.0  # Penalty for errors

def batch_compute_rewards(prompts, responses):
    """Compute rewards for batch of responses"""
    rewards = []
    for prompt, response in zip(prompts, responses):
        # Reset environment for each episode
        obs, info = env.reset()
        reward = compute_reward(response, obs)
        rewards.append(reward)
    return rewards

## 5. Create Training Dataset

In [ ]:
# Generate training prompts from environment
def create_training_dataset(num_episodes=50):
    """Create dataset from environment episodes"""
    prompts = []
    
    for i in range(num_episodes):
        obs, info = env.reset()
        
        # Format prompt
        prompt = f"""Analyze the following text for hallucinations:

Text: {obs['generated_text']}

Task: {obs['task_instruction']}

Provide your analysis:"""
        
        prompts.append(prompt)
        
        if (i + 1) % 10 == 0:
            print(f"Generated {i + 1}/{num_episodes} prompts")
    
    dataset = Dataset.from_dict({"prompt": prompts})
    return dataset

# Create dataset
train_dataset = create_training_dataset(num_episodes=50)
print(f"\n✅ Created dataset with {len(train_dataset)} episodes")
print(f"\nExample prompt:\n{train_dataset[0]['prompt'][:200]}...")

## 6. Configure GRPO Trainer

In [ ]:
# Training configuration
training_args = GRPOConfig(
    output_dir="./hallucination-hunter-qwen",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    warmup_steps=10,
    max_grad_norm=1.0,
    fp16=True,
    report_to="none",  # Disable wandb
)

# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    reward_function=batch_compute_rewards,
)

print("✅ GRPO Trainer configured")

## 7. Train Model

In [ ]:
# Track metrics
training_metrics = {
    'steps': [],
    'rewards': [],
    'losses': []
}

# Train
print("🚀 Starting training...\n")
trainer.train()

print("\n✅ Training complete!")

## 8. Evaluate & Visualize Results

In [ ]:
# Extract training history
history = trainer.state.log_history

# Parse metrics
steps = []
rewards = []
losses = []

for entry in history:
    if 'loss' in entry:
        steps.append(entry.get('step', 0))
        losses.append(entry['loss'])
    if 'reward' in entry:
        rewards.append(entry['reward'])

# Create plots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss curve
if losses:
    axes[0].plot(steps[:len(losses)], losses, 'b-', linewidth=2)
    axes[0].set_xlabel('Training Steps', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training Loss Over Time', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)

# Reward curve
if rewards:
    axes[1].plot(range(len(rewards)), rewards, 'g-', linewidth=2)
    axes[1].set_xlabel('Episodes', fontsize=12)
    axes[1].set_ylabel('Average Reward', fontsize=12)
    axes[1].set_title('Reward Progress', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Plots saved to training_results.png")

## 9. Test Trained Model

In [ ]:
# Test on new episode
obs, info = env.reset()

test_prompt = f"""Analyze the following text for hallucinations:

Text: {obs['generated_text']}

Task: {obs['task_instruction']}

Provide your analysis:"""

# Generate response
FastLanguageModel.for_inference(model)
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Get reward
reward = compute_reward(response, obs)

print("\n" + "="*80)
print("TEST EPISODE")
print("="*80)
print(f"\nPrompt:\n{test_prompt[:200]}...")
print(f"\nModel Response:\n{response[len(test_prompt):]}")
print(f"\nReward: {reward:.3f}")
print(f"Episode ID: {info['episode_id']}")
print(f"Difficulty: {info['difficulty_level']}")

## 10. Save Model

In [ ]:
# Save LoRA adapters
model.save_pretrained("hallucination-hunter-lora")
tokenizer.save_pretrained("hallucination-hunter-lora")

print("✅ Model saved to ./hallucination-hunter-lora")

# Optional: Push to HuggingFace Hub
# model.push_to_hub("your-username/hallucination-hunter-qwen")
# tokenizer.push_to_hub("your-username/hallucination-hunter-qwen")

## 11. Summary Statistics

In [ ]:
# Print summary
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)
print(f"\nModel: Qwen2.5-7B-Instruct (4-bit + LoRA)")
print(f"Training Episodes: {len(train_dataset)}")
print(f"Total Steps: {len(steps)}")

if losses:
    print(f"\nLoss:")
    print(f"  Initial: {losses[0]:.4f}")
    print(f"  Final: {losses[-1]:.4f}")
    print(f"  Improvement: {((losses[0] - losses[-1]) / losses[0] * 100):.1f}%")

if rewards:
    print(f"\nRewards:")
    print(f"  Mean: {np.mean(rewards):.3f}")
    print(f"  Std: {np.std(rewards):.3f}")
    print(f"  Max: {np.max(rewards):.3f}")
    print(f"  Min: {np.min(rewards):.3f}")

print(f"\nEnvironment: {ENV_URL}")
print(f"\n✅ Training complete! Model ready for deployment.")